In [ ]:
import os
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

def setup_deterministic_environment(enable=True):
    """Toggles strict determinism at the CUDA/cuDNN level."""
    if enable:
        os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
        torch.use_deterministic_algorithms(True)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.manual_seed(42)
    else:
        os.environ.pop("CUBLAS_WORKSPACE_CONFIG", None)
        torch.use_deterministic_algorithms(False)
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True
        torch.manual_seed(42)

def run_inference_and_get_logits(model, tokenizer, prompts, batch_size):
    """Runs inference and extracts the logits for the first generated token."""
    inputs = tokenizer(prompts[:batch_size], return_tensors="pt", padding=True).to(model.device)

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=False)

    next_token_logits = outputs.logits[:, -1, :]
    return next_token_logits

def evaluate_determinism(logits_a, logits_b):
    """Calculates the evaluation matrix metrics between two logit tensors."""
    l_a = logits_a[0].float()
    l_b = logits_b[0].float()

    token_a = torch.argmax(l_a).item()
    token_b = torch.argmax(l_b).item()
    exact_match = (token_a == token_b)

    max_diff = torch.max(torch.abs(l_a - l_b)).item()
    mse = F.mse_loss(l_a, l_b).item()

    log_probs_a = F.log_softmax(l_a, dim=-1)
    probs_b = F.softmax(l_b, dim=-1)
    kl_div = F.kl_div(log_probs_a, probs_b, reduction='sum').item()

    return {
        "Exact Token Match": exact_match,
        "Max Absolute Error (L-inf)": f"{max_diff:.8f}",
        "Logit MSE": f"{mse:.8f}",
        "KL Divergence": f"{kl_div:.8f}"
    }

def calculate_exact_match_percentage_over_runs(logits_list):
    """Calculates the percentage of runs that match the top token of the first run."""
    if not logits_list:
        return 0.0

    first_run_top_token = torch.argmax(logits_list[0][0]).item() # Assuming logits_list[i][0] is the logit for the target prompt
    match_count = 0
    for i in range(len(logits_list)):
        if torch.argmax(logits_list[i][0]).item() == first_run_top_token:
            match_count += 1
    return (match_count / len(logits_list)) * 100

def generate_synthetic_ledger(num_transactions=200):
    """Generates a long context string for credit and fraud risk evaluation."""
    torch.manual_seed(42) # Ensure the prompt itself is consistent across runs
    return "\n".join([
        f"TRX_{i}: Amount ${torch.randint(10, 10000, (1,)).item()}, "
        f"Location: {'In State' if i % 2 == 0 else 'Out of State'}, "
        f"Merchant ID: {torch.randint(1000, 9999, (1,)).item()}"
        for i in range(num_transactions)
    ])

def main():
    # Use a small Qwen model for rapid local testing
    model_id = "Qwen/Qwen1.5-0.5B-Chat"
    print(f"Loading {model_id}...\n")

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="auto"
    )

    # Generate the extended prompts to exacerbate floating-point drift
    print("Generating synthetic credit and fraud risk ledger...")
    synthetic_history = generate_synthetic_ledger(200)
    target_prompt = (
        "Analyze the following detailed transaction history and calculate the probability "
        "of an account takeover or fraud event based on velocity and location anomalies.\n"
        f"{synthetic_history}\n"
        "Risk Assessment Output:"
    )
    padding_prompt = "Process the following log data: " + ("SYS_OK 200 " * 500)

    N_RUNS = 100

    print(f"\n--- Testing NON-DETERMINISTIC Baseline over {N_RUNS} runs ---")
    setup_deterministic_environment(enable=False)

    non_det_single_prompt_logits = []

    print(f"Running {N_RUNS} inferences for non-deterministic single prompt...")
    for _ in range(N_RUNS):
        non_det_single_prompt_logits.append(run_inference_and_get_logits(model, tokenizer, [target_prompt], batch_size=1))

    single_prompt_match_percentage_non_det = calculate_exact_match_percentage_over_runs(non_det_single_prompt_logits)

    print(f"Non-Deterministic Single Prompt - Exact Next Token Match Percentage: {single_prompt_match_percentage_non_det:.2f}%")

    print(f"\n--- Testing STRICT DETERMINISM over {N_RUNS} runs ---")
    setup_deterministic_environment(enable=True)

    det_single_prompt_logits = []

    print(f"Running {N_RUNS} inferences for deterministic single prompt...")
    for _ in range(N_RUNS):
        det_single_prompt_logits.append(run_inference_and_get_logits(model, tokenizer, [target_prompt], batch_size=1))

    single_prompt_match_percentage_det = calculate_exact_match_percentage_over_runs(det_single_prompt_logits)

    print(f"Deterministic Single Prompt - Exact Next Token Match Percentage: {single_prompt_match_percentage_det:.2f}%")

if __name__ == "__main__":
    main()

Loading Qwen/Qwen1.5-0.5B-Chat...



Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Generating synthetic credit and fraud risk ledger...

--- Testing NON-DETERMINISTIC Baseline over 100 runs ---
Running 100 inferences for non-deterministic single prompt...


In [1]:
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.models.qwen2.modeling_qwen2 import Qwen2RMSNorm

# ==========================================
# 1. Environment & Seed Setup
# ==========================================
def set_deterministic_environment(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)
    torch.backends.cudnn.deterministic = True
    # We will toggle benchmark during the test to prove our integer math ignores it
    print("Environment seeded and deterministic flags set.")

# ==========================================
# 2. Integer Module Definitions
# ==========================================
class IntegerLinear(nn.Module):
    def __init__(self, in_features, out_features, bias=True, bit_width=8):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.bit_width = bit_width

        self.weight = nn.Parameter(torch.randn(out_features, in_features))
        if bias:
            self.bias = nn.Parameter(torch.randn(out_features))
        else:
            self.register_parameter('bias', None)

    def forward(self, x):
        # Symmetric quantization scaling
        w_scale = self.weight.abs().max() / (2**(self.bit_width-1) - 1)
        x_scale = x.abs().max() / (2**(self.bit_width-1) - 1)

        w_int = (self.weight / w_scale).round().to(torch.int32)
        x_int = (x / x_scale).round().to(torch.int32)

        # Exact integer accumulation
        out_int = torch.matmul(x_int, w_int.t())

        if self.bias is not None:
            b_int = (self.bias / (w_scale * x_scale)).round().to(torch.int32)
            out_int += b_int

        # Rescale back for the next standard block operation
        out = out_int.float() * (w_scale * x_scale)
        return out

class IntegerRMSNorm(nn.Module):
    def __init__(self, hidden_size, eps=1e-6, bit_width=16):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(hidden_size))
        self.eps_int = int(eps * (2**16))
        self.bit_width = bit_width

    def forward(self, x):
        # Convert input to int32 for processing
        x_scale = x.abs().max() / (2**(self.bit_width-1) - 1)
        x_int = (x / x_scale).round().to(torch.int32)

        shift = 8
        x_shifted = x_int >> shift
        variance_int = (x_shifted * x_shifted).mean(dim=-1, keepdim=True)

        # Integer inverse square root approximation
        inv_rms_int = torch.round(1024.0 / torch.sqrt(variance_int.float() + 1.0)).to(torch.int32)
        w_int = (self.weight * 1024).round().to(torch.int32)

        out_int = (x_int * inv_rms_int * w_int) >> 20

        # Rescale back to float for pipeline compatibility
        return out_int.float() * x_scale

class IntegerSiLU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        # Convert to int32
        x_scale = x.abs().max() / 127.0
        x_int = (x / x_scale).round().to(torch.int32)

        three_int = 3 * 128
        six_int = 6 * 128

        # Piecewise Hard-SiLU approximation
        relu6_int = torch.clamp(x_int + three_int, min=0, max=six_int)
        out_int = (x_int * relu6_int) // six_int

        return out_int.float() * x_scale

# ==========================================
# 3. Model Patching Logic
# ==========================================
def harden_model_to_integer(model):
    for name, module in model.named_children():

        # Replace Linear
        if isinstance(module, nn.Linear):
            int_layer = IntegerLinear(module.in_features, module.out_features, bias=(module.bias is not None))
            int_layer.weight.data = module.weight.data
            if module.bias is not None:
                int_layer.bias.data = module.bias.data
            setattr(model, name, int_layer)

        # Replace RMSNorm
        elif isinstance(module, Qwen2RMSNorm):
            int_norm = IntegerRMSNorm(module.weight.shape[0], eps=module.variance_epsilon)
            int_norm.weight.data = module.weight.data
            setattr(model, name, int_norm)

        # Replace Activation (SiLU usually sits inside the MLP block in Qwen2)
        elif name == "act_fn":
            setattr(model, name, IntegerSiLU())

        else:
            # Recursively apply
            harden_model_to_integer(module)

# ==========================================
# 4. Determinism Verification Test
# ==========================================
def run_determinism_test(model, input_text):
    tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen1.5-0.5B")
    inputs = tokenizer(input_text, return_tensors="pt")

    print(f"\nRunning determinism test on input: '{input_text}'")

    # Run 1: Standard benchmark settings
    torch.backends.cudnn.benchmark = True
    with torch.no_grad():
        out1 = model(**inputs).logits

    # Run 2: Disable benchmark (this usually causes floating-point drift)
    torch.backends.cudnn.benchmark = False
    with torch.no_grad():
        out2 = model(**inputs).logits

    # Check for absolute identity
    diff = torch.abs(out1 - out2).max()
    print(f"Maximum bit-level divergence between runs: {diff.item()}")

    if diff == 0:
        print("SUCCESS: The integer implementation is perfectly deterministic.")
    else:
        print("FAILURE: Numerical drift detected.")

    # Optional: Print generation to see the "gibberish" effect from lack of calibration
    # tokens = torch.argmax(out1, dim=-1)
    # print("Sample Output Text:", tokenizer.decode(tokens[0]))

# ==========================================
# 5. Main Execution
# ==========================================
if __name__ == "__main__":
    set_deterministic_environment(42)

    print("Loading base Qwen 1.5-0.5B model...")
    model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen1.5-0.5B")

    print("Converting model to strict Integer Execution Pipeline...")
    harden_model_to_integer(model)
    model.eval()

    run_determinism_test(model, "Determinism in neural networks requires")

Environment seeded and deterministic flags set.
Loading base Qwen 1.5-0.5B model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


Running determinism test on input: 'Determinism in neural networks requires'


RuntimeError: mean(): could not infer output dtype. Input dtype must be either a floating point or complex dtype. Got: Int

To stop the warnings about `HF_TOKEN`, you can create a token on the Hugging Face website and add it to your Colab secrets.

1.  Go to [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) and create a new token with at least 'read' access.
2.  In Colab, click on the "🔑 Secrets" icon in the left sidebar.
3.  Click "Add new secret".
4.  For 'Name', enter `HF_TOKEN` (this is important, it needs to match).
5.  For 'Value', paste your Hugging Face token.
6.  Enable "Notebook access" for this notebook.

Once set, you can configure the `transformers` library to use it as shown below. This will also potentially speed up model downloads and prevent the warning messages.

In [2]:
# Import the necessary library
from google.colab import userdata
from huggingface_hub import login

# Retrieve the HF_TOKEN from Colab secrets
hf_token = userdata.get('HF_TOKEN')

# Log in to Hugging Face Hub (if token is available)
if hf_token:
    login(token=hf_token)
    print("Successfully logged into Hugging Face Hub.")
else:
    print("HF_TOKEN not found in Colab secrets. Model loading will proceed without authentication.")

SecretNotFoundError: Secret HF_TOKEN does not exist.

After running the above cell, you can re-run the previous cell (cell `845b6ea3`) and the warnings about `HF_TOKEN` should no longer appear.